# 06 — Benchmark
So sánh PBE, SCAN, HSE06 và ML-XC trên **sai số năng lượng nguyên tử hóa so với CCSD(T)/cc-pVDZ** (kcal/mol) cho các phân tử test. SCAN và HSE06 được chạy tự hợp (cùng hình học và basis); ML-XC = PBE + ΔE_θ đánh giá trên mật độ PBE (không tự hợp).

- PBE/SCAN/HSE06 còn được báo trên cả 50 phân tử (không huấn luyện gì) để có thống kê lớn hơn; ML-XC chỉ có thể báo trên test.
- Có **resume** cho phần SCF SCAN/HSE06 (cache JSON theo hệ).
- Tổng năng lượng DFT không so với CCSD(T) trực tiếp được (xem bảng độ lệch tổng ở cuối).

**Đầu ra:** `data/processed/benchmark_results.csv`, `benchmark_metrics.json`, `benchmark_parity.png`.

In [ ]:
# Cài thư viện (chỉ trên Colab; runtime Colab bị reset sau mỗi phiên). Chạy cục bộ: pip install -r requirements.txt
import sys
if "google.colab" in sys.modules:
    %pip install -q pyscf flax optax h5py matplotlib

In [ ]:
import os, sys, json, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/ml_xc_colab")
else:                                    # chạy cục bộ (VS Code/Jupyter): notebook nằm trong notebooks/
    here = Path(os.environ.get("ML_XC_ROOT", Path.cwd())).resolve()
    ROOT = next((p for p in (here, here.parent) if (p / "utils").is_dir()), None)
    if ROOT is None:
        raise RuntimeError("Không thấy thư mục utils/. Mở notebook trong ml_xc_colab/notebooks hoặc đặt ML_XC_ROOT.")
sys.path.insert(0, str(ROOT))
ROOT = str(ROOT)
DATA, RAW, PROC, CKPT = (f"{ROOT}/data", f"{ROOT}/data/raw", f"{ROOT}/data/processed", f"{ROOT}/data/checkpoints")
for d in (RAW, PROC, CKPT):
    os.makedirs(d, exist_ok=True)
try:                                     # giới hạn bộ nhớ PySCF = 70% RAM (MB); đặt trước khi import pyscf
    import psutil
    os.environ.setdefault("PYSCF_MAX_MEMORY", str(int(psutil.virtual_memory().total / 1e6 * 0.7)))
except ImportError:
    pass
print("ROOT =", ROOT)

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from utils import dft_utils as dft
from utils import data_utils as du
from utils import model_utils as mu

H5 = f"{PROC}/qm9_xc_dataset.h5"
BENCH = f"{PROC}/benchmark"
os.makedirs(BENCH, exist_ok=True)
BASIS, GRID_LEVEL = "cc-pvdz", 3
XCS = {"scan": "SCAN", "hse06": "HSE06"}
MAX_WALL_H = 10.5

tab = du.energy_table(H5)
systems = pd.read_csv(f"{PROC}/systems.csv").set_index("name").loc[tab.index]
test_mols = [n for n in tab.index if tab.loc[n, "split"] == "test"]
all_mols = [n for n in tab.index if not tab.loc[n, "is_atom"]]
print(f"hệ trong dataset: {len(tab)} | phân tử: {len(all_mols)} | test: {len(test_mols)}")

## 1. SCF tự hợp cho SCAN và HSE06 (mọi hệ, có cache)

In [ ]:
t_start = time.time()
for xc in XCS:
    for n in sorted(tab.index, key=lambda k: tab.loc[k, "n_atoms"]):
        p = f"{BENCH}/{n}_{xc}.json"
        if os.path.exists(p):
            continue
        if (time.time() - t_start) / 3600 > MAX_WALL_H:
            print("Dừng theo giới hạn thời gian; chạy lại để tiếp tục.")
            break
        s = systems.loc[n]
        try:
            mol = dft.build_mol(json.loads(s.symbols_json), json.loads(s.coords_json), basis=BASIS, spin=int(s.spin))
            info, _ = dft.run_ks(mol, xc, grid_level=GRID_LEVEL)
            json.dump(info, open(p, "w"))
            print(f"{xc:6s} {n:14s} E={info['e_tot']:.6f}  conv={info['converged']}  {info['wall_s']}s")
        except Exception as e:
            print(f"{xc:6s} {n:14s} LỖI: {type(e).__name__}: {str(e)[:100]}")
print(f"{(time.time()-t_start)/60:.1f} phút")

In [ ]:
for xc in XCS:
    vals = {n: json.load(open(f"{BENCH}/{n}_{xc}.json")) for n in tab.index if os.path.exists(f"{BENCH}/{n}_{xc}.json")}
    tab[f"e_{xc}"] = pd.Series({n: v["e_tot"] for n, v in vals.items()})
    tab[f"conv_{xc}"] = pd.Series({n: v["converged"] for n, v in vals.items()})
    tab[f"t_{xc}"] = pd.Series({n: v["wall_s"] for n, v in vals.items()})
pbe_t = {n: json.load(open(f"{PROC}/pbe/{n}.json"))["wall_s"] for n in tab.index}
tab["t_pbe"] = pd.Series(pbe_t)
miss = [n for n in tab.index if tab.loc[n, ["e_scan", "e_hse06"]].isna().any()]
print("Hệ thiếu kết quả SCAN/HSE06:", miss if miss else "không")
unconv = [(n, xc) for xc in XCS for n in tab.index if tab.loc[n, f"conv_{xc}"] == False]   # noqa: E712
print("SCF không hội tụ:", unconv if unconv else "không")

## 2. ML-XC trên test (lần đầu tập test được dùng)

In [ ]:
net, params, cfg = mu.load_checkpoint(f"{CKPT}/best_model.msgpack")
print("Mô hình:", cfg["name"], cfg["hidden"], "| val RMSE AE =", round(cfg["best_val"], 3), "kcal/mol")
atoms = [n for n in tab.index if tab.loc[n, "is_atom"]]
ds = mu.make_dataset([du.load_system(H5, n) for n in atoms + test_mols], du.reduced_gradient)
de_fn, _ = mu.make_fns(net)
t0 = time.time()
dE = mu.predict_delta(de_fn, params, ds)
t_ml = (time.time() - t0) / len(ds)
print(f"ΔE_θ cho {len(ds)} hệ: {t_ml*1e3:.1f} ms/hệ (đã gồm biên dịch jit lần đầu)")

In [ ]:
def stat_row(err):
    s = du.error_stats(err); lo, hi = du.bootstrap_mae_ci(err)
    return {"MAE": s["mae"], "MAE_CI95_lo": lo, "MAE_CI95_hi": hi, "RMSE": s["rmse"], "max|err|": s["max"], "n": s["n"]}

rows = {}
rows["PBE"] = stat_row(du.ae_error_kcal(tab, test_mols, "e_pbe"))
rows["SCAN"] = stat_row(du.ae_error_kcal(tab, test_mols, "e_scan"))
rows["HSE06"] = stat_row(du.ae_error_kcal(tab, test_mols, "e_hse06"))
rows["ML-XC (PBE+ΔE_θ)"] = stat_row(du.ae_error_kcal(tab, test_mols, "e_pbe", dE))
res = pd.DataFrame(rows).T
res["chi phí / hệ (s)"] = [tab.loc[test_mols, "t_pbe"].mean(), tab.loc[test_mols, "t_scan"].mean(),
                           tab.loc[test_mols, "t_hse06"].mean(), tab.loc[test_mols, "t_pbe"].mean() + t_ml]
print("Sai số năng lượng nguyên tử hóa so với CCSD(T)/cc-pVDZ trên TEST (kcal/mol):")
res.round(2)

In [ ]:
full = pd.DataFrame({"PBE": stat_row(du.ae_error_kcal(tab, all_mols, "e_pbe")),
                     "SCAN": stat_row(du.ae_error_kcal(tab, all_mols, "e_scan")),
                     "HSE06": stat_row(du.ae_error_kcal(tab, all_mols, "e_hse06"))}).T
print(f"Cả {len(all_mols)} phân tử (các phương pháp này không dùng nhãn nào để huấn luyện), kcal/mol:")
full.round(2)

## 3. Sai số Δ tổng theo nguyên tử, và baseline theo nguyên tố
Baseline `Δ ≈ Σ N_el c_el` (fit trên train + nguyên tử) cho biết mô hình có học được gì ngoài thống kê thành phần hay không. Baseline này khử đúng trên năng lượng nguyên tử hóa (vì nguyên tử nằm trong tập fit), nên chỉ có ý nghĩa ở thước đo tổng theo nguyên tử.

In [ ]:
train_names = [n for n in tab.index if tab.loc[n, "split"] == "train"]
coef = mu.fit_element_baseline(tab, train_names)
base = mu.element_baseline_predict(tab, test_mols, coef)
na = tab.loc[test_mols, "n_atoms"].values
tgt = tab.loc[test_mols, "delta"].values
tot = pd.DataFrame({
    "PBE (Δ=0)": du.error_stats((0 - tgt) / na * 1e3),
    "baseline theo nguyên tố": du.error_stats((np.array([base[n] for n in test_mols]) - tgt) / na * 1e3),
    "ML-XC": du.error_stats((np.array([dE[n] for n in test_mols]) - tgt) / na * 1e3),
}).T[["mae", "rmse", "max", "n"]]
print("Sai số Δ tổng trên mỗi nguyên tử, TEST (mHa/nguyên tử):")
tot.round(3)

## 4. Độ lệch tổng năng lượng (để đối chiếu, không phải thước đo độ chính xác)
DFT và CCSD(T)/cc-pVDZ xử lý tương quan và basis khác nhau, nên tổng năng lượng lệch hàng trăm mHa và tăng theo kích thước.

In [ ]:
off = pd.DataFrame({m: (tab.loc[all_mols, "e_ccsdt"] - tab.loc[all_mols, c]) for m, c in
                    [("PBE", "e_pbe"), ("SCAN", "e_scan"), ("HSE06", "e_hse06")]})
print("E_CCSD(T) − E_DFT trên", len(all_mols), "phân tử (Ha): trung bình / độ lệch chuẩn")
pd.DataFrame({"mean": off.mean(), "std": off.std(), "min": off.min(), "max": off.max()}).round(3)

In [ ]:
errs = {"PBE": du.ae_error_kcal(tab, test_mols, "e_pbe"), "SCAN": du.ae_error_kcal(tab, test_mols, "e_scan"),
        "HSE06": du.ae_error_kcal(tab, test_mols, "e_hse06"), "ML-XC": du.ae_error_kcal(tab, test_mols, "e_pbe", dE)}
out = pd.DataFrame(errs, index=test_mols)
out["formula"] = systems.loc[test_mols, "formula"].values
out.to_csv(f"{PROC}/benchmark_results.csv")
json.dump({"test_AE_error_kcal": res.to_dict("index"), "all_molecules_AE_error_kcal": full.to_dict("index"),
           "total_delta_per_atom_mHa": tot.to_dict("index"), "model": {k: cfg[k] for k in ("name", "hidden", "lr")},
           "basis": BASIS}, open(f"{PROC}/benchmark_metrics.json", "w"), indent=2, default=float)

fig, ax = plt.subplots(figsize=(6, 4))
for k, v in errs.items():
    ax.plot(np.sort(np.abs(v)), np.arange(1, len(v) + 1) / len(v), marker="o", ms=3, label=k)
ax.set_xlabel("|sai số năng lượng nguyên tử hóa| (kcal/mol)"); ax.set_ylabel("phân vị tích lũy"); ax.legend()
plt.tight_layout(); plt.savefig(f"{PROC}/benchmark_parity.png", dpi=120); plt.show()

### Diễn giải
- Test chỉ có ~10 phân tử: xem khoảng tin cậy MAE (CI95) và `max|err|`, đừng chỉ nhìn một con số.
- ML-XC ở đây được chấm theo năng lượng nguyên tử hóa với mật độ PBE (không tự hợp); chưa có kết luận về đạo hàm, hình học hay tự hợp.
- Nhãn CCSD(T)/cc-pVDZ phụ thuộc basis nhỏ nên không chuyển sang basis khác được.